In [ ]:
# Per Namitha & Isaiah's work over summer 2026: fixed misaligned triggers

import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
from pathlib import Path
from collections import Counter
import json

import numpy as np
import pandas as pd
import mne
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from pyprep.find_noisy_channels import NoisyChannels

# --- CONFIG ---
CONFIG = {
    "data_dir": Path(r"C:\Users\isaia\Box\MisophoniaProject_AIISH\DataUpload"),
    "results_dir": Path(r"C:\Users\isaia\Box\MisophoniaProject_AIISH\Results"),
    "cdt_suffix": "_lpp.cdt",
    "eog_channels": ["VEOG", "HEOG"],
    "calib_marker": "32",
    "audio_onset_marker": "1000001",
    "target_sfreq": 250,
    "ref_channels": "average",
    "gap_threshold_sec": 500.0,
    "save_diagnostic_plot": True,
}

EXCLUDE_SUBJECTS = {"118", "120", "128", "143"}

KNOWN_DESCS = {"1", "2", "4", "8", "16", "32", "1000001", "Recording paused"}
TRIGGER_LABELS = {
    1: "Unpleasant",
    2: "Neutral",
    4: "Pleasant",
    8: "Unpleasant_noise",
    16: "Pleasant_noise",
}
TRIGGER_STRS = {str(k) for k in TRIGGER_LABELS}
EXCLUDE_DESCS = {"32", "1000001", "Recording paused"}


# --- HELPER FUNCTIONS ---
def inject_events_to_annotations(raw_obj, events_arr, event_id_dict):
    inv_event_id = {v: k for k, v in event_id_dict.items()}
    onsets_sec = (events_arr[:, 0] - raw_obj.first_samp) / raw_obj.info["sfreq"]
    durations = np.zeros(len(events_arr))
    descriptions = [inv_event_id[code] for code in events_arr[:, 2]]

    new_annot = mne.Annotations(
        onset=onsets_sec,
        duration=durations,
        description=descriptions,
        orig_time=raw_obj.info["meas_date"],
    )
    raw_obj.set_annotations(new_annot)
    return raw_obj


def detect_bad_channels_separately(raw_obj, ransac_sample_prop=0.25, ransac_corr_thresh=0.75, lof_neighbors=10):
    raw_eeg = raw_obj.copy().pick_types(eeg=True, exclude=[])
    eeg_chs = raw_eeg.ch_names

    noisy_detector = NoisyChannels(raw_eeg, random_state=42)
    noisy_detector.find_bad_by_ransac(sample_prop=ransac_sample_prop, corr_thresh=ransac_corr_thresh)
    ransac_bads = [str(ch) for ch in noisy_detector.bad_by_ransac]

    data = raw_eeg.get_data()
    variances = np.var(data, axis=1, keepdims=True)
    psd_means = np.mean(np.abs(np.fft.rfft(data, axis=1)), axis=1, keepdims=True)
    features = np.hstack([np.log(variances + 1e-12), np.log(psd_means + 1e-12)])

    n_neighbors = min(lof_neighbors, max(1, len(eeg_chs) - 1))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination="auto")
    preds = lof.fit_predict(features)
    lof_bads = [eeg_chs[i] for i in range(len(eeg_chs)) if preds[i] == -1]

    union_bads = sorted(set(ransac_bads + lof_bads))

    return {
        "ransac_bads": ransac_bads,
        "lof_bads": lof_bads,
        "union_bads": union_bads,
    }


def extract_calibration_delays(raw, sub_label, gap_threshold_sec):
    onsets = raw.annotations.onset
    descs_clean = [str(d).split(".")[0].strip() for d in raw.annotations.description]

    calib_target = str(CONFIG["calib_marker"]).split(".")[0].strip()
    audio_target = str(CONFIG["audio_onset_marker"]).split(".")[0].strip()

    pair_times, delays_ms, pair_indices = [], [], []
    for i, d in enumerate(descs_clean):
        if d == calib_target:
            for lookahead in range(1, 4):
                if i + lookahead < len(descs_clean) and descs_clean[i + lookahead] == audio_target:
                    t32 = onsets[i]
                    t_audio = onsets[i + lookahead]
                    pair_times.append(t32)
                    delays_ms.append((t_audio - t32) * 1000.0)
                    pair_indices.append((i, i + lookahead))
                    break

    pair_times = np.array(pair_times)
    delays_ms = np.array(delays_ms)

    if len(delays_ms) == 0:
        raise AssertionError(f"No calibration pairs found for {sub_label}.")

    gaps = np.diff(pair_times) if len(pair_times) > 1 else np.array([])
    block_breaks = np.where(gaps > gap_threshold_sec)[0]
    block_ids = np.zeros(len(pair_times), dtype=int)
    for b in block_breaks:
        block_ids[b + 1:] += 1

    raw_std = delays_ms.std()
    z_scores = (delays_ms - delays_ms.mean()) / (raw_std if raw_std > 0 else 1.0)
    outlier_mask = np.abs(z_scores) > 2

    if outlier_mask.any():
        print(f"\n[CALIB OUTLIERS] {sub_label}: Found {outlier_mask.sum()} outlier pairs (ignored in average):")
        for idx in np.where(outlier_mask)[0]:
            annot_32_idx, annot_audio_idx = pair_indices[idx]
            print(
                f"  - Pair #{idx} | Marker indices: ({annot_32_idx} -> {annot_audio_idx}) | "
                f"Onset: {pair_times[idx]:.3f}s | Delay: {delays_ms[idx]:.2f} ms | z-score: {z_scores[idx]:.2f}"
            )
        valid_delays = delays_ms[~outlier_mask]
        clean_mean = float(valid_delays.mean())
        clean_std = float(valid_delays.std())
    else:
        clean_mean = float(delays_ms.mean())
        clean_std = float(delays_ms.std())

    return {
        "descs_clean": descs_clean,
        "delays_ms": delays_ms,
        "pair_times": pair_times,
        "block_ids": block_ids,
        "n_blocks": int(block_ids.max() + 1),
        "n_outliers": int(outlier_mask.sum()),
        "outlier_mask": outlier_mask,
        "mean_delay_ms": clean_mean,
        "std_delay_ms": clean_std,
    }


def save_delay_plot(calib, sub_label, out_dir):
    fig, ax = plt.subplots(figsize=(8, 4))
    delays_ms, block_ids, outlier_mask = calib["delays_ms"], calib["block_ids"], calib["outlier_mask"]
    for b in np.unique(block_ids):
        mask = block_ids == b
        ax.scatter(np.where(mask)[0], delays_ms[mask], label=f"Block {b}")
    ax.axhline(calib["mean_delay_ms"], color="gray", linestyle="--", label="Clean mean (excl. outliers)")
    if outlier_mask.sum() > 0:
        ax.scatter(
            np.where(outlier_mask)[0],
            delays_ms[outlier_mask],
            facecolors="none",
            edgecolors="red",
            s=150,
            linewidths=1.5,
            label="Outlier (>2 SD)",
        )
    ax.set_xlabel("Pair index")
    ax.set_ylabel("Delay (ms)")
    ax.set_title(f"{sub_label}: 32 -> 1000001 delay per calibration pair")
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_dir / f"{sub_label}_calib_delay_diagnostic.png", dpi=150)
    plt.close(fig)


# --- BATCH RUN ---
subject_folders = [f for f in CONFIG["data_dir"].iterdir() if f.is_dir() and f.name.isdigit()]
qc_records = []

for sub_folder in sorted(subject_folders, key=lambda x: int(x.name)):
    sub = sub_folder.name

    if sub in EXCLUDE_SUBJECTS:
        print(f"\n[SKIP] Subject {sub}: Explicitly excluded.")
        continue

    out_dir = CONFIG["results_dir"] / sub
    out_dir.mkdir(parents=True, exist_ok=True)

    # Filename resolution for Subject 139 (ses-2) vs Standard subjects
    if sub == "139":
        sub_label = f"{sub}_ses-2"
        raw_file = sub_folder / f"{sub}_ses-2{CONFIG['cdt_suffix']}"
    else:
        sub_label = sub
        raw_file = sub_folder / f"{sub}{CONFIG['cdt_suffix']}"

    if not raw_file.exists():
        print(f"\n[SKIP] {sub_label}: File not found at {raw_file}")
        continue

    print(f"\n================ PROCESSING {sub_label} ================")
    try:
        raw = mne.io.read_raw_curry(raw_file, preload=True)
        sfreq = raw.info["sfreq"]

        # Marker counts and condition checks
        descs_clean_all = [str(d).split(".")[0].strip() for d in raw.annotations.description]
        marker_counts = Counter(descs_clean_all)
        print(f"[MARKERS] Raw marker frequencies: {dict(marker_counts)}")

        present_conditions = {c for c in TRIGGER_STRS if c in marker_counts}
        missing_conditions = set(TRIGGER_STRS) - present_conditions
        if missing_conditions:
            missing_labels = [f"{c} ({TRIGGER_LABELS[int(c)]})" for c in sorted(missing_conditions, key=int)]
            print(f"[WARNING] {sub_label} is MISSING conditions: {', '.join(missing_labels)}")

        unexpected_codes = set(descs_clean_all) - KNOWN_DESCS
        if unexpected_codes:
            print(f"[WARNING] {sub_label}: unexpected markers {unexpected_codes}")

        type_map = {ch: "eog" for ch in CONFIG["eog_channels"] if ch in raw.ch_names}
        if type_map:
            raw.set_channel_types(type_map)

        montage = raw.get_montage()
        montage_missing = []
        if montage is not None:
            eeg_names = [ch for ch, t in zip(raw.ch_names, raw.get_channel_types()) if t == "eeg"]
            montage_missing = [ch for ch in eeg_names if ch not in montage.ch_names]
        else:
            print(f"[WARNING] {sub_label}: no digitized montage detected.")

        calib = extract_calibration_delays(raw, sub_label, CONFIG["gap_threshold_sec"])
        if CONFIG["save_diagnostic_plot"]:
            save_delay_plot(calib, sub_label, out_dir)

        mean_delay_ms = calib["mean_delay_ms"]
        delay_sec = mean_delay_ms / 1000.0
        descs_clean = calib["descs_clean"]
        onsets = raw.annotations.onset

        exp_mask = np.array([d in TRIGGER_STRS for d in descs_clean])
        exp_onsets = onsets[exp_mask]
        exp_descs = np.array(descs_clean)[exp_mask]

        codes = np.array([int(d) for d in exp_descs])
        corrected_samples = np.round((exp_onsets + delay_sec) * sfreq).astype(int) + raw.first_samp
        events_corrected = np.column_stack([
            corrected_samples, np.zeros(len(corrected_samples), dtype=int), codes
        ])
        event_id = {TRIGGER_LABELS[c]: c for c in sorted(set(codes))}

        raw_resampled, events_resampled = raw.copy().resample(
            CONFIG["target_sfreq"], npad="auto", events=events_corrected
        )

        raw_resampled = inject_events_to_annotations(raw_resampled, events_resampled, event_id)
        bad_ch_results = detect_bad_channels_separately(raw_resampled)
        raw_resampled.info["bads"] = bad_ch_results["union_bads"]

        if CONFIG["ref_channels"] == "average":
            raw_resampled.set_eeg_reference("average", projection=False)
        elif isinstance(CONFIG["ref_channels"], list):
            raw_resampled.set_eeg_reference(CONFIG["ref_channels"])

        fif_path = out_dir / f"{sub_label}_preprocessed_raw.fif"
        events_path = out_dir / f"{sub_label}_events.npy"
        event_id_path = out_dir / f"{sub_label}_event_id.json"
        bad_ch_path = out_dir / f"{sub_label}_bad_channels.json"

        raw_resampled.save(fif_path, overwrite=True)
        np.save(events_path, events_resampled)
        with open(event_id_path, "w") as f:
            json.dump({k: int(v) for k, v in event_id.items()}, f, indent=2)
        with open(bad_ch_path, "w") as f:
            json.dump(bad_ch_results, f, indent=2)

        qc_records.append({
            "subject": sub_label,
            "status": "SUCCESS",
            "orig_sfreq": sfreq,
            "calib_pairs_found": len(calib["delays_ms"]),
            "calib_blocks": calib["n_blocks"],
            "calib_outliers": calib["n_outliers"],
            "mean_delay_ms": round(mean_delay_ms, 2),
            "events_kept": len(events_resampled),
            "conditions_present": ", ".join(sorted(present_conditions, key=int)),
            "conditions_missing": ", ".join(sorted(missing_conditions, key=int)) if missing_conditions else "None",
            "ransac_bads": ", ".join(bad_ch_results["ransac_bads"]),
            "lof_bads": ", ".join(bad_ch_results["lof_bads"]),
            "union_bads": ", ".join(bad_ch_results["union_bads"]),
            "n_bad_channels": len(bad_ch_results["union_bads"]),
            "montage_missing": ", ".join(montage_missing) if montage_missing else "None",
            "unexpected_codes": ", ".join(unexpected_codes) if unexpected_codes else "None",
            "error_msg": None,
        })
        print(f"[SUCCESS] {sub_label}: Done. Bad channels (union): {bad_ch_results['union_bads']}")

    except Exception as e:
        print(f"[ERROR] {sub_label} failed: {e}")
        qc_records.append({
            "subject": sub_label,
            "status": "FAILED",
            "orig_sfreq": None,
            "calib_pairs_found": 0,
            "calib_blocks": None,
            "calib_outliers": None,
            "mean_delay_ms": None,
            "events_kept": 0,
            "conditions_present": "",
            "conditions_missing": "",
            "ransac_bads": "",
            "lof_bads": "",
            "union_bads": "",
            "n_bad_channels": 0,
            "montage_missing": "",
            "unexpected_codes": "",
            "error_msg": str(e),
        })

df_qc = pd.DataFrame(qc_records)
df_qc.to_csv(CONFIG["results_dir"] / "pipeline_qc_summary.csv", index=False)
print("\nBatch processing complete. Summary saved to Results/pipeline_qc_summary.csv")


================ PROCESSING 102 ================
Reading 0 ... 5658367  =      0.000 ...  2762.874 secs...
Leaving device<->head transform as None (no landmarks found)
[MARKERS] Raw marker frequencies: {'1': 40, '1000001': 2364, '4': 39, '16': 40, '8': 40, '2': 39}
[ERROR] 102 failed: No calibration pairs found for 102.

================ PROCESSING 103 ================


C:\Users\isaia\AppData\Local\Temp\ipykernel_33504\2923164231.py:225: RuntimeWarning: The unit for channel(s) HEOG, VEOG has changed from NA to V.
  raw.set_channel_types(type_map)


[ERROR] 103 failed: Could not open data file

================ PROCESSING 104 ================
Reading 0 ... 6123775  =      0.000 ...  2990.124 secs...
Leaving device<->head transform as None (no landmarks found)
[MARKERS] Raw marker frequencies: {'32': 60, '1000001': 2663, '4': 41, '1': 41, '2': 40, '16': 40, '8': 40}


C:\Users\isaia\AppData\Local\Temp\ipykernel_33504\2923164231.py:225: RuntimeWarning: The unit for channel(s) HEOG, VEOG has changed from NA to V.
  raw.set_channel_types(type_map)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Executing RANSAC
This may take a while, so be patient...


...


RANSAC done!
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Overwriting existing file.
Writing C:\Users\isaia\Box\MisophoniaProject_AIISH\Results\202\202_preprocessed_raw.fif
Overwriting existing file.
Closing C:\Users\isaia\Box\MisophoniaProject_AIISH\Results\202\202_preprocessed_raw.fif
[done]
[SUCCESS] 202: Done. Bad channels (union): ['CP4', 'CPz', 'F12', 'FC4', 'P4']

================ PROCESSING 213 ================
Reading 0 ... 5746431  =      0.000 ...  2805.874 secs...
Leaving device<->head transform as None (no landmarks found)
[MARKERS] Raw marker frequencies: {'32': 60, '1000001': 2000, '4': 41, '1': 41, '2': 40, '16': 40, '8': 40, 'Recording paused': 3}

[CALIB OUTLIERS] 213: Found 3 outlier pairs (ignored in average):
  - Pair #15 | Marker indices: (30 -> 31) | Onset: 25.988s | Delay: 62.50 ms | z-score: -2.48
  - Pair #24 | Marker indices: (49 -> 50) | Onset: 39.489s | Delay: 62.50 ms | z-score: -2.48
  - P

C:\Users\isaia\AppData\Local\Temp\ipykernel_33504\2923164231.py:225: RuntimeWarning: The unit for channel(s) HEOG, VEOG has changed from NA to V.
  raw.set_channel_types(type_map)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Executing RANSAC
This may take a while, so be patient...


  0%|          |  : 0/561 [00:00<?,       ?it/s]


RANSAC done!
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Overwriting existing file.
Writing C:\Users\isaia\Box\MisophoniaProject_AIISH\Results\213\213_preprocessed_raw.fif
Overwriting existing file.
Closing C:\Users\isaia\Box\MisophoniaProject_AIISH\Results\213\213_preprocessed_raw.fif
[done]
[SUCCESS] 213: Done. Bad channels (union): ['CPz', 'Cz', 'Pz']

Batch processing complete. Summary saved to Results/pipeline_qc_summary.csv


In [3]:
qc_file = Path(r"C:\Users\isaia\Box\MisophoniaProject_AIISH\Results\pipeline_qc_summary.csv")
df_qc = pd.read_csv(qc_file)
df_qc

,subject,status,orig_sfreq,calib_pairs_found,calib_blocks,calib_outliers,mean_delay_ms,events_kept,conditions_present,conditions_missing,ransac_bads,lof_bads,union_bads,n_bad_channels,montage_missing,unexpected_codes,error_msg
0,102,FAILED,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,No calibration pairs found for 102.
1,103,FAILED,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,Could not open data file
2,104,SUCCESS,2048.00021,60,2.0,0.0,69.64,202,"1, 2, 4, 8, 16",NaN,"Cz, CPz","Fp1, Fp2, Fz, FCz, Cz, CP3, CPz, Pz","CP3, CPz, Cz, FCz, Fp1, Fp2, Fz, Pz",8,NaN,NaN,NaN
3,105,SUCCESS,2048.00021,60,2.0,0.0,68.98,202,"1, 2, 4, 8, 16",NaN,"F8, F12, Cz, T8, CPz","Fz, FCz, Cz, T8, CPz","CPz, Cz, F12, F8, FCz, Fz, T8",7,NaN,NaN,NaN
4,106,SUCCESS,2048.00021,30,1.0,1.0,68.12,201,"1, 2, 4, 8, 16",NaN,"Cz, CPz, Pz",CPz,"CPz, Cz, Pz",3,NaN,NaN,NaN
5,107,SUCCESS,2048.00021,60,2.0,0.0,70.05,202,"1, 2, 4, 8, 16",NaN,"Cz, CPz","Fp2, F12, FT11, CPz, CP4, P4, P8","CP4, CPz, Cz, F12, FT11, Fp2, P4, P8",8,NaN,NaN,NaN
6,108,SUCCESS,2048.00021,60,2.0,4.0,71.32,202,"1, 2, 4, 8, 16",NaN,"F12, Cz, CPz",F8,"CPz, Cz, F12, F8",4,NaN,NaN,NaN
7,109,SUCCESS,2048.00021,60,2.0,2.0,68.38,202,"1, 2, 4, 8, 16",NaN,"Cz, CPz",Fp2,"CPz, Cz, Fp2",3,NaN,NaN,NaN
8,110,SUCCESS,2048.00021,60,2.0,1.0,69.03,202,"1, 2, 4, 8, 16",NaN,"Cz, CPz","Fp2, CP3, CP4, M2, P4","CP3, CP4, CPz, Cz, Fp2, M2, P4",7,NaN,NaN,NaN
9,112,SUCCESS,2048.00021,60,2.0,6.0,70.87,202,"1, 2, 4, 8, 16",NaN,"Cz, CPz","Fz, Cz, T8, CPz","CPz, Cz, Fz, T8",4,NaN,NaN,NaN
